# 02 — Nettoyage (data cleaning)

## Objectif
Nettoyer le dataset pour obtenir une donnée cohérente et exploitable.

### Règles choisies 
- Remplacer les valeurs “sales” (UNKNOWN, ERROR, etc.) par des valeurs manquantes (NaN).
- Convertir les colonnes numériques en nombres.
- Recalculer Total Spent = Quantity * Price Per Unit (car c’est une valeur dérivée).
- Supprimer les lignes sans Item (transaction inutilisable).
- Supprimer les lignes sans Transaction Date (car on veut une analyse temporelle par mois).
- Remplacer les valeurs manquantes de Payment Method et Location par la valeur la plus fréquente (mode), pour éviter de perdre trop de lignes.

In [1]:
import pandas as pd
import numpy as np

In [2]:
raw_file_path = "../DATA/RAW/dirty_cafe_sales.csv"

raw_data = pd.read_csv(raw_file_path)

print("Dataset brut (lignes, colonnes) :", raw_data.shape)
raw_data.head(5)

Dataset brut (lignes, colonnes) : (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
clean_data = raw_data.copy()

In [4]:
dirty_values = ["UNKNOWN", "ERROR", "NaN", "nan", ""]

clean_data = clean_data.replace(dirty_values, np.nan)

text_columns = clean_data.select_dtypes(include="object").columns
for col in text_columns:
    clean_data[col] = clean_data[col].astype(str).str.strip()
    clean_data[col] = clean_data[col].replace(["nan"], np.nan)  

In [5]:
clean_data["Quantity"] = pd.to_numeric(clean_data["Quantity"], errors="coerce")
clean_data["Price Per Unit"] = pd.to_numeric(clean_data["Price Per Unit"], errors="coerce")
clean_data["Total Spent"] = pd.to_numeric(clean_data["Total Spent"], errors="coerce")

In [6]:
clean_data["Total Spent"] = clean_data["Quantity"] * clean_data["Price Per Unit"]

In [7]:
clean_data["Transaction Date"] = pd.to_datetime(clean_data["Transaction Date"], errors="coerce")

In [8]:
rows_before = len(clean_data)

clean_data = clean_data.dropna(subset=["Item"])

clean_data = clean_data.dropna(subset=["Transaction Date"])

clean_data = clean_data.dropna(subset=["Quantity", "Price Per Unit", "Total Spent"])

clean_data = clean_data[(clean_data["Quantity"] > 0) & (clean_data["Price Per Unit"] > 0)]

rows_after = len(clean_data)

print("Lignes supprimées au total :", rows_before - rows_after)
print("Dataset nettoyé (lignes, colonnes) :", clean_data.shape)

Lignes supprimées au total : 2227
Dataset nettoyé (lignes, colonnes) : (7773, 8)


In [9]:
payment_mode = clean_data["Payment Method"].mode(dropna=True)[0]
location_mode = clean_data["Location"].mode(dropna=True)[0]

clean_data["Payment Method"] = clean_data["Payment Method"].fillna(payment_mode)
clean_data["Location"] = clean_data["Location"].fillna(location_mode)

print("Payment Method rempli avec :", payment_mode)
print("Location remplie avec :", location_mode)

Payment Method rempli avec : Digital Wallet
Location remplie avec : In-store


In [10]:
print("Types des colonnes :")
print(clean_data.dtypes)

print("\nManquants par colonne (top) :")
missing_final = clean_data.isna().sum().sort_values(ascending=False)
print(missing_final[missing_final > 0])

print("\nManquants Transaction Date :", clean_data["Transaction Date"].isna().sum())
print("Valeurs 'Unknown' Payment Method :", (clean_data["Payment Method"] == "Unknown").sum())
print("Valeurs 'Unknown' Location :", (clean_data["Location"] == "Unknown").sum())

Types des colonnes :
Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object

Manquants par colonne (top) :
Series([], dtype: int64)

Manquants Transaction Date : 0
Valeurs 'Unknown' Payment Method : 0
Valeurs 'Unknown' Location : 0


In [11]:
output_file_path = "../DATA/PROCESSED/clean_cafe_sales.csv"
clean_data.to_csv(output_file_path, index=False)

print("CSV final exporté :", output_file_path)

CSV final exporté : ../DATA/PROCESSED/clean_cafe_sales.csv
